In [6]:
# Step 1: Import required libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix


In [7]:
# Step 2: Set paths and parameters
train_dir = "dataset/train"
valid_dir = "dataset/valid"

IMG_SIZE = (224, 224)   # VGG16 required size
BATCH_SIZE = 32
EPOCHS = 10


In [8]:
# Step 3: Create data generators with augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False  # important for evaluation
)


Found 2540 images belonging to 2 classes.
Found 100 images belonging to 2 classes.


In [9]:
# Step 4: Load VGG16 without top layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all convolutional layers
for layer in base_model.layers:
    layer.trainable = False


In [10]:
# Step 5: Add custom classification layers
x = Flatten()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)  # binary classification

model = Model(inputs=base_model.input, outputs=output)


In [11]:
# Step 6: Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,137,729 (80.63 MB)

 Trainable params: 6,423,041 (24.50 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [12]:
# Step 7: Train the model
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=valid_generator
)



/Users/user89/Desktop/ASD_v/venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


2025-09-21 12:00:13.437629: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


80/80 ━━━━━━━━━━━━━━━━━━━━ 25s 304ms/step - accuracy: 0.6205 - loss: 0.9175 - val_accuracy: 0.7300 - val_loss: 0.5559
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 24s 300ms/step - accuracy: 0.6697 - loss: 0.8796 - val_accuracy: 0.7200 - val_loss: 0.5661
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 24s 303ms/step - accuracy: 0.6689 - loss: 0.9349 - val_accuracy: 0.7200 - val_loss: 0.6331
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 24s 303ms/step - accuracy: 0.6783 - loss: 0.9037 - val_accuracy: 0.7100 - val_loss: 0.5604
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 24s 300ms/step - accuracy: 0.6941 - loss: 0.8792 - val_accuracy: 0.7300 - val_loss: 0.6887
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 24s 301ms/step - accuracy: 0.6894 - loss: 0.8856 - val_accuracy: 0.7000 - val_loss: 0.5668
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 25s 308ms/step - accuracy: 0.7224 - loss: 0.8285 - val_accuracy: 0.7400 - val_loss: 0.5743
Epoch 8/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 25s 305ms/step - accuracy: 0.7114 - loss: 0.8143 - val_accuracy: 0.740

In [13]:
# Step 8: Evaluate the model
loss, accuracy = model.evaluate(valid_generator)
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 201ms/step - accuracy: 0.7200 - loss: 0.5929
Validation Loss: 0.5929
Validation Accuracy: 0.7200


In [23]:
# Step 9: Get predictions for evaluation
y_pred = model.predict(valid_generator)
y_pred_classes = (y_pred > 0.5).astype(int)

print(classification_report(valid_generator.classes, y_pred_classes, target_names=['Autistic', 'Non_Autistic']))

cm = confusion_matrix(valid_generator.classes, y_pred_classes)
print("Confusion Matrix:\n", cm)


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 217ms/step
              precision    recall  f1-score   support

    Autistic       0.85      0.56      0.67        50
Non_Autistic       0.67      0.90      0.77        50

    accuracy                           0.73       100
   macro avg       0.76      0.73      0.72       100
weighted avg       0.76      0.73      0.72       100

Confusion Matrix:
 [[28 22]
 [ 5 45]]


In [14]:
model.save("autism_detection_vgg16.h5")
print("autism_detection_vgg16.h5")


autism_detection_vgg16.h5


In [15]:
import numpy as np
from tensorflow.keras.preprocessing import image
import tensorflow as tf

# Load your trained model
model = tf.keras.models.load_model("autism_detection_vgg16.h5")

# Function to predict autism or not
def predict_autism(img_path):
    IMG_SIZE = (224, 224)  # Same as training
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0  # normalize
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

    prediction = model.predict(img_array)[0][0]

    if prediction > 0.5:
        print(f"{img_path} --> Non_Autistic ({prediction:.4f})")
    else:
        print(f"{img_path} --> Autistic ({prediction:.4f})")

# Example usage:
# If you upload a file via Kaggle's "Add data" or `!cp` from input
test_image_path = "dataset/train/non_autistic/Non_Autistic.1002.jpg"
predict_autism(test_image_path)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 623ms/step
dataset/train/non_autistic/Non_Autistic.1002.jpg --> Non_Autistic (0.9958)


### Hyperparameter tuning 

In [16]:
import tensorflow as tf

# Load trained model
model = tf.keras.models.load_model("autism_detection_vgg16.h5")

# Unfreeze layers whose name starts with 'block5'
for layer in model.layers:
    if layer.name.startswith("block5"):
        layer.trainable = True
    else:
        layer.trainable = False  # Keep others frozen

print("Trainable status of layers:")
for layer in model.layers:
    print(f"{layer.name}: {layer.trainable}")


Trainable status of layers:
input_layer_1: False
block1_conv1: False
block1_conv2: False
block1_pool: False
block2_conv1: False
block2_conv2: False
block2_pool: False
block3_conv1: False
block3_conv2: False
block3_conv3: False
block3_pool: False
block4_conv1: False
block4_conv2: False
block4_conv3: False
block4_pool: False
block5_conv1: True
block5_conv2: True
block5_conv3: True
block5_pool: True
flatten_1: False
dense_2: False
dropout_1: False
dense_3: False


In [17]:
from tensorflow.keras.optimizers import Adam

# Recompile with a lower learning rate to avoid damaging pretrained weights
model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Fine-tuning uses very small LR
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,137,729 (80.63 MB)

 Trainable params: 7,079,424 (27.01 MB)

 Non-trainable params: 14,058,305 (53.63 MB)

In [18]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    "dataset/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

valid_generator = valid_datagen.flow_from_directory(
    "dataset/valid",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)


Found 2540 images belonging to 2 classes.
Found 100 images belonging to 2 classes.


In [19]:
history_finetune = model.fit(
    train_generator,
    epochs=10,  # You can try fewer if overfitting
    validation_data=valid_generator
)


Epoch 1/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 30s 358ms/step - accuracy: 0.7047 - loss: 0.8959 - val_accuracy: 0.7300 - val_loss: 0.6332
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 28s 354ms/step - accuracy: 0.7256 - loss: 0.8072 - val_accuracy: 0.7500 - val_loss: 0.6279
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 28s 354ms/step - accuracy: 0.7327 - loss: 0.7806 - val_accuracy: 0.7400 - val_loss: 0.6647
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 29s 355ms/step - accuracy: 0.7425 - loss: 0.7716 - val_accuracy: 0.7700 - val_loss: 0.6658
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 29s 357ms/step - accuracy: 0.7469 - loss: 0.7866 - val_accuracy: 0.7600 - val_loss: 0.5704
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 28s 358ms/step - accuracy: 0.7531 - loss: 0.7442 - val_accuracy: 0.7500 - val_loss: 0.6174
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 29s 357ms/step - accuracy: 0.7465 - loss: 0.7209 - val_accuracy: 0.7900 - val_loss: 0.5398
Epoch 8/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 30s 371ms/step - accuracy: 0.7626 - loss: 0.6789 - val_accu

In [20]:
loss, accuracy = model.evaluate(valid_generator)
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 310ms/step - accuracy: 0.7900 - loss: 0.5700
Validation Loss: 0.5700
Validation Accuracy: 0.7900


In [21]:
model.save("autism_detection_vgg16_finetuned.h5")
print("Fine-tuned model saved to autism_detection_vgg16_finetuned.h5")


Fine-tuned model saved to autism_detection_vgg16_finetuned.h5


In [22]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Get predictions
y_true = valid_generator.classes
y_pred_prob = model.predict(valid_generator)
y_pred = (y_pred_prob > 0.5).astype(int).reshape(-1)

# Labels
labels = list(valid_generator.class_indices.keys())

# Report
print(classification_report(y_true, y_pred, target_names=labels))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 219ms/step
              precision    recall  f1-score   support

    Autistic       0.80      0.78      0.79        50
Non_Autistic       0.78      0.80      0.79        50

    accuracy                           0.79       100
   macro avg       0.79      0.79      0.79       100
weighted avg       0.79      0.79      0.79       100

Confusion Matrix:
[[39 11]
 [10 40]]


In [7]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

# Load fine-tuned model
model = tf.keras.models.load_model("autism_detection_vgg16_finetuned.h5")

# Function to predict
def predict_autism(img_path):
    IMG_SIZE = (224, 224)
    
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0  # normalize
    img_array = tf.expand_dims(img_array, axis=0)  # add batch dimension

    prediction = model.predict(img_array, verbose=0)[0][0]
    if prediction > 0.5:
        print(f"Prediction: Non_Autistic ({prediction:.2f} confidence)")
    else:
        print(f"Prediction: Autistic ({1 - prediction:.2f} confidence)")


# Example usage
predict_autism("dataset/test/non_autistic/Non_Autistic.26 11.44.29.jpg")  # change path to your image


Prediction: Non_Autistic (0.51 confidence)


In [8]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import os
from sklearn.metrics import accuracy_score

# Load model
model = tf.keras.models.load_model("autism_detection_vgg16_finetuned.h5")

def evaluate_model(test_dir, img_size=(224, 224)):
    y_true = []
    y_pred = []
    
    # Assuming your test directory structure is:
    # test/
    #   autistic/
    #   non_autistic/
    
    for class_name in ['autistic', 'non_autistic']:
        class_dir = os.path.join(test_dir, class_name)
        class_label = 0 if class_name == 'autistic' else 1
        
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(class_dir, img_name)
                
                # Load and preprocess image
                img = image.load_img(img_path, target_size=img_size)
                img_array = image.img_to_array(img) / 255.0
                img_array = tf.expand_dims(img_array, axis=0)
                
                # Predict
                prediction = model.predict(img_array, verbose=0)[0][0]
                predicted_class = 1 if prediction > 0.5 else 0
                
                y_true.append(class_label)
                y_pred.append(predicted_class)
    
    accuracy = accuracy_score(y_true, y_pred)
    print(f"Test Accuracy: {accuracy:.4f}")
    return accuracy, y_true, y_pred

# Usage
test_accuracy, y_true, y_pred = evaluate_model("dataset/test")
print(f"Model Accuracy: {test_accuracy:.2%}")

Test Accuracy: 0.8533
Model Accuracy: 85.33%
